In [1]:
import pandas as pd
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

from patches import compute_patches

In [2]:
# Search image
file = "../data/query/h24a.jpg"

q = Image.open(file)
q = np.array(q, dtype=np.uint32)
q = q.reshape( -1, 3)
q[(q < 20).all(axis=1), :] = 0
q = q[:, 0] + 256 * q[:, 1] + 256 ** 2 * q[:, 2] 


Uq, Cq = np.unique(q, return_counts=True, axis=0)
total_color_pixels = sum(Cq[Uq != 0])
Cq = Cq / total_color_pixels
qcounts = pd.DataFrame(np.c_[Uq, Cq], columns=["code", "count"]).sort_values("count", ascending=False)
qcounts

,code,count
0,0.0,50.866696
19583,15791604.0,0.000956
20124,16777214.0,0.000625
19666,15923190.0,0.000552
19556,15725811.0,0.000552
...,...,...
7503,6382706.0,0.000037
7504,6382944.0,0.000037
7505,6382984.0,0.000037
7506,6383237.0,0.000037


In [ ]:
%%time
patch_size = 224
window_shift = 112

output_dir = os.path.join("..", "data", "artwork")
#images = []
files = list(glob.glob(os.path.join(output_dir, "**")))
random.shuffle(files)
scores = {}
for i, file in enumerate(tqdm(files)):
    s = Image.open(file)
    s = np.array(s, dtype=np.uint32)
    s = s.reshape(-1, 3)
    s[(s < 20).all(axis=1), :] = 0
    s = s[:, 0] + 256 * s[:, 1] + 256 ** 2 * s[:, 2]

    s[~np.isin(s, q)] = 0

    Us, Cs = np.unique(s, return_counts=True, axis=0)
    total_color_pixels = sum(Cs[Us != 0])
    if total_color_pixels == 0:
        continue
    Cs = Cs / total_color_pixels
    scounts = pd.DataFrame(np.c_[Us, Cs], columns=["code", "count"]).sort_values("count", ascending=False)
    overlap = qcounts.merge(scounts, on="code", suffixes=("_q", "_s"), how="left").fillna(0)
    overlap["diff"] = overlap["count_q"] - overlap["count_s"]
    rms = np.sqrt(sum(overlap.query("code != 0")["diff"] ** 2))

    scores[file] = rms

In [ ]:
scores = pd.Series(scores).sort_values()
scores.head(64)

In [ ]:
f, ax = plt.subplots(ncols=4, nrows=40, figsize=(20, 200))
for i, file in enumerate(scores.index[:4*40]):
    col = i // 4
    row = i % 4
    s = Image.open(file)
    s = np.array(s)
    ax[col, row].imshow(s)
    ax[col, row].axes.get_xaxis().set_ticks([])
    ax[col, row].axes.get_yaxis().set_ticks([])
